# 03 - Experiment Results Analysis

Load and compare results from FL experiments.
- Convergence curves (CER vs round)
- Algorithm comparison (FedAvg vs SCAFFOLD vs FedOPT)
- PEFT communication efficiency
- Per-client fairness analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import glob

sns.set_theme(style='whitegrid')

RESULTS_DIR = Path('../experiments/results')

In [ ]:
# Discover all experiment results
experiments = {}
for exp_dir in sorted(RESULTS_DIR.iterdir()):
    if exp_dir.is_dir():
        csv_path = exp_dir / 'metrics.csv'
        summary_path = exp_dir / 'summary.json'
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            summary = {}
            if summary_path.exists():
                with open(summary_path) as f:
                    summary = json.load(f)
            experiments[exp_dir.name] = {'metrics': df, 'summary': summary}
            print(f'{exp_dir.name}: {len(df)} rounds logged')

print(f'\nTotal experiments found: {len(experiments)}')

In [ ]:
# Convergence curves: CER vs round for all experiments
fig, ax = plt.subplots(figsize=(12, 6))

for name, data in experiments.items():
    df = data['metrics']
    if 'global_cer' in df.columns and 'round' in df.columns:
        ax.plot(df['round'], df['global_cer'], label=name, marker='o', markersize=3)

ax.set_xlabel('Communication Round')
ax.set_ylabel('Global CER')
ax.set_title('Convergence: CER vs Communication Round')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Communication efficiency: CER vs cumulative communication (MB)
fig, ax = plt.subplots(figsize=(12, 6))

for name, data in experiments.items():
    df = data['metrics']
    if 'global_cer' in df.columns and 'cumulative_comm_mb' in df.columns:
        ax.plot(df['cumulative_comm_mb'], df['global_cer'], label=name, marker='o', markersize=3)

ax.set_xlabel('Cumulative Communication (MB)')
ax.set_ylabel('Global CER')
ax.set_title('Communication Efficiency: CER vs Total Data Transferred')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Fairness: worst-client CER vs round
fig, ax = plt.subplots(figsize=(12, 6))

for name, data in experiments.items():
    df = data['metrics']
    if 'worst_client_cer' in df.columns and 'round' in df.columns:
        ax.plot(df['round'], df['worst_client_cer'], label=name, marker='s', markersize=3)

ax.set_xlabel('Communication Round')
ax.set_ylabel('Worst Client CER')
ax.set_title('Fairness: Worst-Client CER vs Round')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Summary table
rows = []
for name, data in experiments.items():
    s = data['summary']
    df = data['metrics']
    final_cer = df['global_cer'].iloc[-1] if 'global_cer' in df.columns and len(df) > 0 else None
    rows.append({
        'Experiment': name,
        'Algorithm': s.get('algorithm', '?'),
        'PEFT': s.get('peft_method', '?'),
        'Rounds': s.get('total_rounds', '?'),
        'Final CER': f'{final_cer:.4f}' if final_cer is not None else '?',
        'Total Comm (MB)': f"{s.get('total_mb', '?'):.1f}" if 'total_mb' in s else '?',
        'Time (s)': f"{s.get('elapsed_seconds', '?'):.0f}" if 'elapsed_seconds' in s else '?',
    })

summary_df = pd.DataFrame(rows)
summary_df